# Experimentos de Métricas de Recomendação

Este notebook executa avaliações quantitativas (RMSE, MAE, Hit@K, MRR) sobre o sistema de recomendação, usando os dados reais do projeto.

---

In [19]:
import json
import math
import os
import random
from collections import Counter, defaultdict
import pandas as pd

## Funções utilitárias para carregar dados

In [20]:
def load_line_json(path):
    with open(path, 'r', encoding='utf-8') as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            yield json.loads(line)

def load_reviews(path):
    user_ratings = defaultdict(dict)
    item_ratings = defaultdict(dict)
    for record in load_line_json(path):
        user_id = record.get('user_id')
        item_id = record.get('business_id')
        rating = record.get('stars')
        if user_id is None or item_id is None or rating is None:
            continue
        rating = float(rating)
        user_ratings[user_id][item_id] = rating
        item_ratings[item_id][user_id] = rating
    return user_ratings, item_ratings

def load_business_categories(path):
    categories = {}
    for record in load_line_json(path):
        item_id = record.get('business_id')
        raw_categories = record.get('categories') or ''
        categories[item_id] = {
            part.strip().lower()
            for part in raw_categories.split(',')
            if part.strip()
        }
    return categories

## Funções de avaliação e recomendação

In [21]:
def build_user_norms(user_ratings):
    return {
        user_id: math.sqrt(sum(value * value for value in ratings.values()))
        for user_id, ratings in user_ratings.items()
    }

def cosine_similarity(a, b, norm_a, norm_b):
    if norm_a == 0 or norm_b == 0:
        return 0.0
    common = set(a).intersection(b)
    if not common:
        return 0.0
    numerator = sum(a[item] * b[item] for item in common)
    return numerator / (norm_a * norm_b)

def split_train_test(user_ratings, min_ratings=5, test_ratio=0.2, seed=42):
    random.seed(seed)
    train = {}
    test = []
    for user_id, ratings in user_ratings.items():
        if len(ratings) < min_ratings:
            train[user_id] = ratings.copy()
            continue
        items = list(ratings.items())
        n_test = max(1, int(len(items) * test_ratio))
        test_items = random.sample(items, n_test)
        test_map = {item: rating for item, rating in test_items}
        train[user_id] = {item: rating for item, rating in items if item not in test_map}
        for item, rating in test_items:
            test.append((user_id, item, rating))
    return train, test

In [22]:
def global_mean(train_user_ratings):
    total = 0.0
    count = 0
    for ratings in train_user_ratings.values():
        for rating in ratings.values():
            total += rating
            count += 1
    return total / count if count else 0.0

def item_popularity_scores(train_user_ratings):
    item_counts = Counter()
    item_sum = Counter()
    for ratings in train_user_ratings.values():
        for item, rating in ratings.items():
            item_counts[item] += 1
            item_sum[item] += rating
    return {item: item_sum[item] / item_counts[item] for item in item_counts}

## Funções de predição e ranking

In [23]:
def predict_rating(user_id, item_id, train_user_ratings, user_norms, top_n=50, min_similarity=0.0, similarity_power=1.0):
    target_ratings = train_user_ratings.get(user_id)
    if not target_ratings:
        return None
    target_norm = user_norms.get(user_id, 0.0)
    if target_norm == 0.0:
        return None
    similarities = []
    for other_user, other_ratings in train_user_ratings.items():
        if other_user == user_id:
            continue
        rating = other_ratings.get(item_id)
        if rating is None:
            continue
        sim = cosine_similarity(target_ratings, other_ratings, target_norm, user_norms.get(other_user, 0.0))
        if sim <= min_similarity:
            continue
        similarities.append((sim, rating))
    if not similarities:
        return None
    similarities.sort(key=lambda x: x[0], reverse=True)
    similarities = similarities[:top_n]
    numerator = 0.0
    denominator = 0.0
    for sim, rating in similarities:
        weight = sim ** similarity_power
        numerator += weight * rating
        denominator += weight
    if denominator == 0.0:
        return None
    return numerator / denominator

In [24]:
def rank_candidates(user_id, train_user_ratings, user_norms, candidates, top_n=50, min_similarity=0.0, similarity_power=1.0, max_rank=100):
    scores = []
    rated = set(train_user_ratings.get(user_id, {}))
    for item_id in candidates:
        if item_id in rated:
            continue
        predicted = predict_rating(user_id, item_id, train_user_ratings, user_norms, top_n=top_n, min_similarity=min_similarity, similarity_power=similarity_power)
        if predicted is not None:
            scores.append((predicted, item_id))
    scores.sort(key=lambda x: x[0], reverse=True)
    return [item for _, item in scores[:max_rank]]

## Funções de avaliação quantitativa

In [25]:
def evaluate_predictions(test_set, train_user_ratings, user_norms, global_avg):
    mse = 0.0
    mae = 0.0
    count = 0
    missing = 0
    for user_id, item_id, actual in test_set:
        predicted = predict_rating(user_id, item_id, train_user_ratings, user_norms)
        if predicted is None:
            predicted = global_avg
            missing += 1
        error = predicted - actual
        mse += error * error
        mae += abs(error)
        count += 1
    if count == 0:
        return None
    return {
        'rmse': math.sqrt(mse / count),
        'mae': mae / count,
        'missing_fraction': missing / count,
    }

In [26]:
def evaluate_topk(test_set, train_user_ratings, user_norms, candidates, k=10, **kwargs):
    hits = 0
    rr_sum = 0.0
    total = 0
    for user_id, item_id, _ in test_set:
        ranked = rank_candidates(user_id, train_user_ratings, user_norms, candidates, max_rank=k, **kwargs)
        total += 1
        if item_id in ranked:
            hits += 1
            rr_sum += 1.0 / (ranked.index(item_id) + 1)
    return {
        'hit_rate': hits / total if total else 0.0,
        'mrr': rr_sum / total if total else 0.0,
        'precision_at_k': hits / total / k if total else 0.0,
        'recall_at_k': hits / total if total else 0.0,
    }

## Carregando dados e executando experimentos

In [27]:
import os
data_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "..", "data"))
review_path = os.path.join(data_dir, 'review_final.json')
business_path = os.path.join(data_dir, 'business_final.json')

print("review_path:", review_path)
print("business_path:", business_path)

user_ratings, item_ratings = load_reviews(review_path)
categories = load_business_categories(business_path)
print(f'Usuários: {len(user_ratings)}, Itens: {len(item_ratings)}')

train_user_ratings, test_set = split_train_test(user_ratings, min_ratings=5, test_ratio=0.2)
user_norms = build_user_norms(train_user_ratings)
global_avg = global_mean(train_user_ratings)
popularity_scores = item_popularity_scores(train_user_ratings)
candidates = sorted(popularity_scores, key=lambda item: popularity_scores[item], reverse=True)
print(f'Treino: {len(train_user_ratings)}, Teste: {len(test_set)}')

review_path: c:\projetos\isep\data\review_final.json
business_path: c:\projetos\isep\data\business_final.json


FileNotFoundError: [Errno 2] No such file or directory: 'c:\\projetos\\isep\\data\\review_final.json'

## Avaliação quantitativa: RMSE, MAE, Hit@K, MRR

Testando diferentes parâmetros de vizinhança e similaridade.

In [ ]:
param_grid = [
    {'top_n': 10, 'min_similarity': 0.0, 'similarity_power': 1.0},
    {'top_n': 20, 'min_similarity': 0.0, 'similarity_power': 1.0},
    {'top_n': 50, 'min_similarity': 0.0, 'similarity_power': 1.0},
    {'top_n': 20, 'min_similarity': 0.1, 'similarity_power': 1.0},
    {'top_n': 20, 'min_similarity': 0.2, 'similarity_power': 1.0},
    {'top_n': 20, 'min_similarity': 0.0, 'similarity_power': 2.0},
]
results = []
for params in param_grid:
    pred_eval = evaluate_predictions(test_set, train_user_ratings, user_norms, global_avg)
    topk_eval_5 = evaluate_topk(test_set, train_user_ratings, user_norms, candidates, k=5, **params)
    topk_eval_10 = evaluate_topk(test_set, train_user_ratings, user_norms, candidates, k=10, **params)
    results.append({
        **params,
        **pred_eval,
        'precision@5': topk_eval_5['precision_at_k'],
        'recall@5': topk_eval_5['recall_at_k'],
        'mrr@5': topk_eval_5['mrr'],
        'precision@10': topk_eval_10['precision_at_k'],
        'recall@10': topk_eval_10['recall_at_k'],
        'mrr@10': topk_eval_10['mrr'],
    })
import pandas as pd
df = pd.DataFrame(results)
df

## Análise crítica dos resultados

- RMSE e MAE avaliam a qualidade da predição de rating, mas dependem da quantidade de exemplos de teste por usuário.
- Hit@K e MRR avaliam a qualidade do ranking, ou seja, se o item correto aparece entre os primeiros recomendados.
- A fração de missing mostra quantos casos não tiveram vizinhos suficientes para prever, usando fallback para a média global.
- Parâmetros como top_n e min_similarity afetam cobertura e precisão: mais vizinhos aumentam cobertura, mas podem trazer ruído; thresholds altos reduzem falsos positivos, mas podem perder acertos.
- Compare sempre com o baseline de popularidade: se o modelo não supera o ranking por popularidade, precisa de ajustes ou mais dados.